# Telugu ASR — File 4 of 5
## Multi-Dataset Preparation: MSCIL + Shrutilipi + OpenSLR-66

**Prerequisites (outputs from Files 1 & 2):**
- `./telugu_asr_clean_dataset/` — cleaned MSCIL HuggingFace dataset
- `./telugu_wav2vec2_processor/` — existing processor (kept as backup)

**What this file does:**
- Loads MSCIL (already prepared) + Shrutilipi Telugu (AI4Bharat/AIR news) + OpenSLR-66 (optional)
- Applies the same Telugu text normalization as File 1 across all sources
- Merges into a single HuggingFace dataset and filters by duration
- Builds a new unified vocabulary and processor

**Outputs:**
- `./telugu_multi_dataset/` — merged dataset (~650+ hours)
- `./vocab_v2.json` — updated vocabulary
- `./telugu_wav2vec2_processor_v2/` — new processor for File 5

In [10]:
# ═══════════════════════════════════════════════════════════════
# SECTION 1 — INSTALL DEPENDENCIES
# ═══════════════════════════════════════════════════════════════
%pip install -q "datasets<3.0"
%pip install -q transformers torch evaluate jiwer accelerate soundfile
%pip install -q pyctcdecode  # KenLM decoder — used in File 5

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [10]:
# ═══════════════════════════════════════════════════════════════
# SECTION 2 — IMPORTS & HUGGINGFACE LOGIN
# ═══════════════════════════════════════════════════════════════
import os
import re
import json
import unicodedata
import logging
from pathlib import Path

import numpy as np
from datasets import (
    load_from_disk, load_dataset,
    concatenate_datasets, Audio, Dataset,
)
from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
)
from huggingface_hub import login

# Suppress noisy third-party loggers
logging.getLogger("numba").setLevel(logging.WARNING)
logging.getLogger("datasets").setLevel(logging.WARNING)
logging.getLogger("transformers").setLevel(logging.WARNING)

# ── HuggingFace auth ──────────────────────────────────────────
# Shrutilipi requires a HuggingFace account (free) to download.
# Set your token before running:
#   Linux/macOS : export HF_TOKEN=hf_your_token
#   Windows CMD : set HF_TOKEN=hf_your_token
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Logged in to HuggingFace Hub.")
else:
    print("[WARN] No HF_TOKEN env var found.")
    print("If Shrutilipi download fails, set: export HF_TOKEN=hf_your_token")

[WARN] No HF_TOKEN env var found.
If Shrutilipi download fails, set: export HF_TOKEN=hf_your_token


In [11]:
# ═══════════════════════════════════════════════════════════════
# SECTION 3 — PATHS, CONFIG & TEXT NORMALIZER
# ═══════════════════════════════════════════════════════════════

# ── Input paths (from Files 1 & 2) ───────────────────────────
MSCIL_PATH      = "./telugu_asr_clean_dataset/"
SLR66_PATH      = "./openslr66/"   # optional — see Section 6

# ── Output paths (used by File 5) ────────────────────────────
OUTPUT_DATASET  = "./telugu_multi_dataset/"
OUTPUT_VOCAB    = "./vocab_v2.json"
OUTPUT_PROCESSOR = "./telugu_wav2vec2_processor_v2/"

# ── Audio config ─────────────────────────────────────────────
SAMPLE_RATE     = 16_000
DURATION_MIN    = 1.0    # seconds — same as File 1
DURATION_MAX    = 15.0

# ── Telugu text normalizer ────────────────────────────────────
# Identical to File 1: keep only Telugu Unicode block U+0C00–U+0C7F
# and single spaces. Strip everything else.
_TELUGU_RE = re.compile(r"[^\u0C00-\u0C7F\s]")

def normalize_telugu(text: str) -> str:
    text = unicodedata.normalize("NFC", text)
    text = _TELUGU_RE.sub("", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Smoke test
_t = "కచ్చితంగా! చూపిస్తుంది 123 abc"
print(f"Normalize test: '{_t}'")
print(f"           → '{normalize_telugu(_t)}'")

Normalize test: 'కచ్చితంగా! చూపిస్తుంది 123 abc'
           → 'కచ్చితంగా చూపిస్తుంది'


In [12]:
# ═══════════════════════════════════════════════════════════════
# SECTION 4 — LOAD MSCIL (FILE 1 OUTPUT)
# ═══════════════════════════════════════════════════════════════
assert Path(MSCIL_PATH).exists(), (
    f"MSCIL dataset not found at '{MSCIL_PATH}'.\n"
    "Run File 1 (01_data_exploration_and_cleaning.ipynb) first."
)

mscil = load_from_disk(MSCIL_PATH)
mscil = mscil.cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))
mscil = mscil.add_column("source", ["mscil"] * len(mscil))

print(f"MSCIL loaded     : {len(mscil):>7,} samples")
print(f"Columns          : {mscil.column_names}")
print(f"Sample text      : {mscil['transcription'][0]}")

MSCIL loaded     :  35,426 samples
Columns          : ['audio', 'transcription', 'audio_id', 'source']
Sample text      : కచ్చితంగా చూపిస్తుంది కదా మరి


In [16]:

# ═══════════════════════════════════════════════════════════════
# SECTION 5 — LOAD SHRUTILIPI TELUGU (AI4BHARAT / AIR NEWS)
# ═══════════════════════════════════════════════════════════════
# Source: All India Radio news broadcasts, ~500+ hours Telugu
# HuggingFace: ai4bharat/Shrutilipi, config "telugu"
# License: CC-BY-4.0
# Schema: audio_filepath (Audio), text, duration, lang

def _normalize_columns(ds: Dataset) -> Dataset:
    """Rename Shrutilipi columns → our schema: audio, transcription, audio_id."""
    # audio: Shrutilipi uses "audio_filepath" instead of "audio"
    if "audio_filepath" in ds.column_names and "audio" not in ds.column_names:
        ds = ds.rename_column("audio_filepath", "audio")

    # transcription
    for alt in ("text", "sentence", "normalized_text"):
        if alt in ds.column_names and "transcription" not in ds.column_names:
            ds = ds.rename_column(alt, "transcription")
            break
    assert "transcription" in ds.column_names, (
        f"Cannot find text column. Available: {ds.column_names}"
    )

    # audio_id
    if "audio_id" not in ds.column_names:
        ds = ds.add_column("audio_id",
                           [f"shrutilipi_{i:08d}" for i in range(len(ds))])

    # drop everything except the three columns we need
    keep = {"audio", "transcription", "audio_id"}
    drop = [c for c in ds.column_names if c not in keep]
    if drop:
        ds = ds.remove_columns(drop)
    return ds


shrutilipi = None
try:
    print("Downloading Shrutilipi Telugu (~3–8 GB, first run only)...")
    _ds = load_dataset(
        "ai4bharat/Shrutilipi",
        "telugu",
        split="train",
        trust_remote_code=True,
    )
    print(f"Raw columns      : {_ds.column_names}")

    # Rename columns first, THEN cast audio to 16 kHz
    _ds = _normalize_columns(_ds)
    _ds = _ds.cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))

    _ds = _ds.map(
        lambda b: {"transcription": normalize_telugu(b["transcription"])},
        desc="Normalizing Shrutilipi text",
    )
    _ds = _ds.filter(lambda b: len(b["transcription"]) > 0)
    _ds = _ds.add_column("source", ["news"] * len(_ds))
    shrutilipi = _ds
    print(f"Shrutilipi loaded: {len(shrutilipi):>7,} samples")
    print(f"Sample text      : {shrutilipi['transcription'][0]}")
except Exception as e:
    print(f"[WARN] Shrutilipi load failed: {e}")
    print("Continuing with MSCIL only. Set HF_TOKEN if this is an auth error.")


Raw columns      : ['audio_filepath', 'text', 'duration', 'lang']
Shrutilipi loaded:  56,082 samples
Sample text      : శివన్ ప్రయోగం ప్రక్రియపై శాస్రవేత్తలతో సమావేశాన్ని నిర్వహించారు


In [18]:

# ═══════════════════════════════════════════════════════════════
# SECTION 6 — LOAD OPENSLR-66 (OPTIONAL — CROWDSOURCED TELUGU)
# ═══════════════════════════════════════════════════════════════
# OpenSLR-66: Crowdsourced high-quality Telugu multi-speaker speech
# Download: https://www.openslr.org/66/
# Expected layout after setup:
#   ./openslr66/line_index.tsv   (columns: utt_id \t transcription)
#   ./openslr66/wavs/            (WAV files named <utt_id>.wav)

openslr66 = None

if Path(SLR66_PATH).exists():
    try:
        # Find metadata TSV
        _tsv = next(
            (Path(SLR66_PATH) / n for n in
             ("line_index.tsv", "utt_spk_text.tsv", "line_index_all.tsv")
             if (Path(SLR66_PATH) / n).exists()),
            None,
        )
        assert _tsv is not None, "No TSV metadata found in ./openslr66/"

        rows = []
        with open(_tsv, encoding="utf-8") as _f:
            for line in _f:
                parts = line.strip().split("\t")
                if len(parts) < 2:
                    continue
                utt_id = parts[0]
                text   = parts[-1]
                _wav = Path(SLR66_PATH) / "wavs" / f"{utt_id}.wav"
                if not _wav.exists():
                    _wav = Path(SLR66_PATH) / f"{utt_id}.wav"
                if _wav.exists():
                    rows.append({
                        "audio_id"      : utt_id,
                        "audio"         : str(_wav),   # path string → cast to Audio
                        "transcription" : text,
                    })

        print(f"OpenSLR-66 TSV rows matched to WAVs: {len(rows)}")
        _ds66 = Dataset.from_list(rows)

        # Cast path string → Audio (datasets<3.0 compatible — no path_column arg)
        _ds66 = _ds66.cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))

        _ds66 = _ds66.map(
            lambda b: {"transcription": normalize_telugu(b["transcription"])},
            desc="Normalizing OpenSLR-66 text",
        )
        _ds66 = _ds66.filter(lambda b: len(b["transcription"]) > 0)
        _ds66 = _ds66.add_column("source", ["crowd"] * len(_ds66))
        openslr66 = _ds66
        print(f"OpenSLR-66 loaded: {len(openslr66):>7,} samples")
    except Exception as e:
        print(f"[WARN] OpenSLR-66 load failed: {e}")
else:
    print("OpenSLR-66 directory not found — skipping.")
    print("Download from https://www.openslr.org/66/ to include it.")


OpenSLR-66 TSV rows matched to WAVs: 4448


Flattening the indices: 100%|██████████| 4448/4448 [00:00<00:00, 36330.02 examples/s]

OpenSLR-66 loaded:   4,448 samples


In [19]:
# ═══════════════════════════════════════════════════════════════
# SECTION 7 — MERGE DATASETS & DURATION FILTER
# ═══════════════════════════════════════════════════════════════
parts = [mscil]
if shrutilipi is not None:
    parts.append(shrutilipi)
if openslr66 is not None:
    parts.append(openslr66)

merged = concatenate_datasets(parts)
print(f"Before filter    : {len(merged):>7,} samples total")
for _src in ("mscil", "news", "crowd"):
    _n = sum(1 for s in merged["source"] if s == _src)
    if _n:
        print(f"  {_src:8s}       : {_n:>7,}")

# Compute duration (seconds) — same sentinel approach as File 1
_SENTINEL = -1.0

def _add_duration(batch):
    try:
        arr = batch["audio"]["array"]
        sr  = batch["audio"]["sampling_rate"]
        return {"duration": float(len(arr) / sr)}
    except Exception:
        return {"duration": _SENTINEL}

merged = merged.map(_add_duration, desc="Computing durations")
before = len(merged)
merged = merged.filter(
    lambda b: DURATION_MIN <= b["duration"] <= DURATION_MAX,
    desc="Duration filter",
)
merged = merged.remove_columns(["duration"])

print(f"After filter     : {len(merged):>7,} samples (removed {before - len(merged):,})")
print("Source breakdown after filter:")
for _src in ("mscil", "news", "crowd"):
    _n = sum(1 for s in merged["source"] if s == _src)
    if _n:
        print(f"  {_src:8s}       : {_n:>7,}")

Before filter    :  95,956 samples total
  mscil          :  35,426
  news           :  56,082
  crowd          :   4,448


Duration filter: 100%|██████████| 95956/95956 [00:55<00:00, 1742.17 examples/s]


After filter     :  91,951 samples (removed 4,005)
Source breakdown after filter:
  mscil          :  35,426
  news           :  52,077
  crowd          :   4,448


In [20]:
# ═══════════════════════════════════════════════════════════════
# SECTION 8 — BUILD VOCABULARY
# ═══════════════════════════════════════════════════════════════
# Collect all unique characters across ALL datasets.
# Column-wise access avoids decoding audio.

all_chars: set = set()
for text in merged["transcription"]:
    all_chars.update(set(text.replace(" ", "")))

sorted_chars = sorted(all_chars)
vocab: dict  = {char: idx for idx, char in enumerate(sorted_chars)}
vocab["|"]     = len(vocab)   # word boundary token
vocab["[UNK]"] = len(vocab)   # unknown character
vocab["[PAD]"] = len(vocab)   # CTC blank / padding — MUST be last

with open(OUTPUT_VOCAB, "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False, indent=2)

print(f"Vocabulary saved : {OUTPUT_VOCAB}")
print(f"  Total tokens   : {len(vocab)}")
print(f"  Telugu chars   : {len(sorted_chars)}")
print(f"  Special tokens : '|'→{vocab['|']}  '[UNK]'→{vocab['[UNK]']}  '[PAD]'→{vocab['[PAD]']}")
print(f"  PAD token id   : {vocab['[PAD]']}  (CTC blank token)")

Vocabulary saved : ./vocab_v2.json
  Total tokens   : 69
  Telugu chars   : 66
  Special tokens : '|'→66  '[UNK]'→67  '[PAD]'→68
  PAD token id   : 68  (CTC blank token)


In [21]:
# ═══════════════════════════════════════════════════════════════
# SECTION 9 — BUILD PROCESSOR & SAVE
# ═══════════════════════════════════════════════════════════════

tokenizer = Wav2Vec2CTCTokenizer(
    OUTPUT_VOCAB,
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
)
feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=SAMPLE_RATE,
    padding_value=0.0,
    do_normalize=True,          # zero-mean / unit-variance per utterance
    return_attention_mask=True,
)
processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
)
processor.save_pretrained(OUTPUT_PROCESSOR)

print(f"Processor saved  : {OUTPUT_PROCESSOR}")
print(f"Vocab size       : {processor.tokenizer.vocab_size}")

# Sanity check: encode → decode must be lossless
_test = "కచ్చితంగా చూపిస్తుంది"
_ids  = processor.tokenizer(_test).input_ids
_dec  = processor.tokenizer.decode(_ids)
assert _dec == _test, f"Round-trip FAIL: '{_dec}' != '{_test}'"
print("Round-trip test  : PASS")

Processor saved  : ./telugu_wav2vec2_processor_v2/
Vocab size       : 69
Round-trip test  : PASS


In [22]:
# ═══════════════════════════════════════════════════════════════
# SECTION 10 — SAVE MERGED DATASET & VERIFY
# ═══════════════════════════════════════════════════════════════

# Drop 'source' column — not needed by Trainer
final_dataset = merged.remove_columns(["source"])
final_dataset.save_to_disk(OUTPUT_DATASET)

# Verify by reloading
_check = load_from_disk(OUTPUT_DATASET)
assert len(_check) == len(final_dataset), "Save/reload mismatch!"

print("=" * 55)
print("FILE 4 COMPLETE")
print("=" * 55)
print(f"  Dataset   : {OUTPUT_DATASET}")
print(f"  Samples   : {len(final_dataset):,}")
print(f"  Columns   : {final_dataset.column_names}")
print(f"  Vocab     : {OUTPUT_VOCAB}  ({processor.tokenizer.vocab_size} tokens)")
print(f"  Processor : {OUTPUT_PROCESSOR}")
print()
print("Ready for File 5 (05_improved_full_training.ipynb).")

Saving the dataset (30/30 shards): 100%|██████████| 91951/91951 [00:24<00:00, 3755.22 examples/s]

FILE 4 COMPLETE
  Dataset   : ./telugu_multi_dataset/
  Samples   : 91,951
  Columns   : ['audio', 'transcription', 'audio_id']
  Vocab     : ./vocab_v2.json  (69 tokens)
  Processor : ./telugu_wav2vec2_processor_v2/

Ready for File 5 (05_improved_full_training.ipynb).
